# D1 adaptive attacker — do PCA / noise / channel-drop hold under encoder fine-tune?

Symmetry experiment to D2/D3 adaptive. Re-runs the encoder-fine-tune attack against three D1-defended EEGNets (PCA k=8, noise σ=1.0, channel-drop k=8). Each defense is applied to the windows before the EEGNet sees them; the attacker then warm-starts from the trained EEGNet weights and fine-tunes end-to-end on subject-id.

If D1's privacy claim (~0.30-0.36 generic top-1) collapses back toward the no-defense A1 baseline (~0.41) under fine-tune, then D1 — like D2 — only worked against generic attackers. If it holds, D1's input-transform defense is genuinely robust at this level.

Send back `results/23_d1_adaptive_attacker.json`.

**Runtime → L4 GPU.** Expected wall ~50-55 min (3 EEGNet trainings + 3 fine-tunes).

**Don't `Save a copy in GitHub` from Colab.**

## 1. Clone repo

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. Prefetch PhysioNet imagery (~2 min)

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

## 5. Run experiment

Expected wall: ~50-55 min (3 EEGNet trainings + 3 fine-tunes).

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.23_d1_adaptive_attacker --all

## 6. Emit run metadata + result JSON

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
EXPERIMENT = "experiments.23_d1_adaptive_attacker"
ARGS = ['--all']
RESULTS = 'results/23_d1_adaptive_attacker.json'
EXTRA_OUTPUTS = []
TAG = "d1_adaptive"
try:
    git_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR).decode().strip()
except Exception as exc:
    print(f'  git unavailable ({exc}); using "unknown"'); git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS,
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS] + EXTRA_OUTPUTS}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f: json.dump(meta, f, indent=2)
print('=== run metadata ==='); print(json.dumps(meta, indent=2)); print()
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing — run cell did not finish. Re-run the run cell, then this cell.')
print(f'=== {RESULTS} ==='); print(open(RESULTS).read())


## 7. Download artifacts

In [ ]:
from google.colab import files
files.download('results/23_d1_adaptive_attacker.json')

files.download(f'runs/{run_id}/meta.json')
